Exercice 1 — Hyperparameter Tuning

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# ============================================================
# DONNÉES
# ============================================================
X, y = make_circles(1000, noise=0.03, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================================
# MODÈLE BASELINE (sans tuning)
# ============================================================
def build_model(lr=0.01, n_layers=1, n_neurons=16):
    """Construit un modèle avec les hyperparamètres donnés"""
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(2,)))

    # Ajouter n_layers couches cachées
    for _ in range(n_layers):
        model.add(tf.keras.layers.Dense(n_neurons,
                                         activation='relu'))

    # Couche de sortie
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Baseline
print(" BASELINE")
baseline = build_model(lr=0.01, n_layers=1, n_neurons=16)
baseline.fit(X_train, y_train,
             epochs=50, verbose=0,
             validation_split=0.1)
_, acc_baseline = baseline.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy baseline : {acc_baseline*100:.2f}%")

# ============================================================
# GRID SEARCH MANUEL
# ============================================================
print("\n GRID SEARCH — Test de toutes les combinaisons\n")

# Hyperparamètres à tester
learning_rates = [0.001, 0.01]
n_layers_list  = [1, 2, 3]
n_neurons_list = [16, 32, 64]

results = []

for lr in learning_rates:
    for n_layers in n_layers_list:
        for n_neurons in n_neurons_list:
            model = build_model(lr, n_layers, n_neurons)
            model.fit(X_train, y_train,
                      epochs=50,
                      verbose=0,
                      validation_split=0.1)
            _, acc = model.evaluate(X_test, y_test, verbose=0)
            results.append({
                'lr': lr,
                'layers': n_layers,
                'neurons': n_neurons,
                'accuracy': acc
            })
            print(f"lr={lr} | layers={n_layers} | "
                  f"neurons={n_neurons:3d} → {acc*100:.2f}%")

# Meilleure combinaison
best = max(results, key=lambda x: x['accuracy'])
print(f"\n MEILLEURE COMBINAISON :")
print(f"   Learning rate : {best['lr']}")
print(f"   Couches       : {best['layers']}")
print(f"   Neurones      : {best['neurons']}")
print(f"   Accuracy      : {best['accuracy']*100:.2f}%")
print(f"   Gain vs baseline : +{(best['accuracy']-acc_baseline)*100:.2f}%")

# ============================================================
# RÉSUMÉ
# ============================================================
print(f"""
 RÉSUMÉ — Impact du Hyperparameter Tuning :

Le tuning a permis de passer de {acc_baseline*100:.1f}% à {best['accuracy']*100:.1f}%.
Le learning rate contrôle la vitesse d'apprentissage.
Plus de couches permet au réseau d'apprendre des patterns complexes.
Plus de neurones augmente la capacité du modèle mais risque l'overfitting.
Le Grid Search teste toutes les combinaisons mais peut être lent sur de grands espaces.
""")

 BASELINE
Accuracy baseline : 100.00%

 GRID SEARCH — Test de toutes les combinaisons

lr=0.001 | layers=1 | neurons= 16 → 75.50%
lr=0.001 | layers=1 | neurons= 32 → 86.00%
lr=0.001 | layers=1 | neurons= 64 → 95.50%
lr=0.001 | layers=2 | neurons= 16 → 99.50%
lr=0.001 | layers=2 | neurons= 32 → 100.00%
lr=0.001 | layers=2 | neurons= 64 → 100.00%
lr=0.001 | layers=3 | neurons= 16 → 100.00%
lr=0.001 | layers=3 | neurons= 32 → 100.00%
lr=0.001 | layers=3 | neurons= 64 → 100.00%
lr=0.01 | layers=1 | neurons= 16 → 100.00%
lr=0.01 | layers=1 | neurons= 32 → 100.00%
lr=0.01 | layers=1 | neurons= 64 → 100.00%
lr=0.01 | layers=2 | neurons= 16 → 100.00%
lr=0.01 | layers=2 | neurons= 32 → 100.00%
lr=0.01 | layers=2 | neurons= 64 → 100.00%
lr=0.01 | layers=3 | neurons= 16 → 92.00%
lr=0.01 | layers=3 | neurons= 32 → 100.00%
lr=0.01 | layers=3 | neurons= 64 → 93.50%

 MEILLEURE COMBINAISON :
   Learning rate : 0.001
   Couches       : 2
   Neurones      : 32
   Accuracy      : 100.00%
   Gain vs base

Exercice 2 — Ensemble Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

# ============================================================
# DONNÉES
# ============================================================
X, y = make_circles(1000, noise=0.03, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================================
# ENTRAÎNER 3 MODÈLES DIFFÉRENTS
# ============================================================

# Modèle 1 : Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
acc_dt = accuracy_score(y_test, dt.predict(X_test))

# Modèle 2 : Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))

# Modèle 3 : Neural Network
nn = build_model(lr=0.001, n_layers=2, n_neurons=32)
nn.fit(X_train, y_train, epochs=100, verbose=0)
nn_probs = nn.predict(X_test, verbose=0).ravel()
acc_nn = accuracy_score(y_test, (nn_probs >= 0.5).astype(int))

print(f"Decision Tree  : {acc_dt*100:.2f}%")
print(f"Random Forest  : {acc_rf*100:.2f}%")
print(f"Neural Network : {acc_nn*100:.2f}%")

# ============================================================
# ENSEMBLE PAR VOTE MAJORITAIRE
# ============================================================
pred_dt = dt.predict(X_test)
pred_rf = rf.predict(X_test)
pred_nn = (nn_probs >= 0.5).astype(int)

# Vote majoritaire : on prend la classe qui a le plus de votes
votes = pred_dt + pred_rf + pred_nn   # somme des votes
ensemble_pred = (votes >= 2).astype(int)  # majoritaire si >= 2

acc_ensemble = accuracy_score(y_test, ensemble_pred)
print(f"\n Ensemble (vote majoritaire) : {acc_ensemble*100:.2f}%")

# ============================================================
# ENSEMBLE PONDÉRÉ (weighted voting)
# ============================================================
# On donne plus de poids aux modèles plus précis
w_dt = acc_dt
w_rf = acc_rf
w_nn = acc_nn

weighted_votes = (w_dt * pred_dt +
                  w_rf * pred_rf +
                  w_nn * pred_nn)

total_weight = w_dt + w_rf + w_nn
weighted_pred = (weighted_votes / total_weight >= 0.5).astype(int)

acc_weighted = accuracy_score(y_test, weighted_pred)
print(f"  Ensemble pondéré           : {acc_weighted*100:.2f}%")

# ============================================================
# TABLEAU COMPARATIF
# ============================================================
print("\n" + "="*40)
print("COMPARAISON FINALE")
print("="*40)
models = {
    'Decision Tree' : acc_dt,
    'Random Forest' : acc_rf,
    'Neural Network': acc_nn,
    'Ensemble Vote' : acc_ensemble,
    'Ensemble Pond.': acc_weighted
}
for name, acc in models.items():
    bar = "=" * int(acc * 30)
    print(f"{name:15s} : {acc*100:.2f}%  {bar}")

print(f"""
POURQUOI L'ENSEMBLE PERFORME MIEUX ?

Chaque modèle fait des erreurs différentes.
Quand on combine leurs votes, les erreurs s'annulent.
C'est comme demander l'avis de plusieurs experts
plutôt que de faire confiance à un seul.
Le vote pondéré donne plus d'importance aux meilleurs modèles.
""")

Exercice 3 — Transfer Learning

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# DONNÉES — CIFAR-10
# (10 classes : avion, voiture, oiseau, chat, cerf,
#  chien, grenouille, cheval, bateau, camion)
# ============================================================
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normaliser
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# MobileNet attend des images 32x32 minimum
# On prend un sous-ensemble pour aller plus vite
x_train_small = x_train[:5000]
y_train_small = y_train[:5000]
x_test_small  = x_test[:1000]
y_test_small  = y_test[:1000]

# One-hot encoding
y_train_enc = to_categorical(y_train_small, 10)
y_test_enc  = to_categorical(y_test_small,  10)

print(f"Train : {x_train_small.shape}")
print(f"Test  : {x_test_small.shape}")

# Noms des classes CIFAR-10
class_names = ['avion', 'voiture', 'oiseau', 'chat',
               'cerf', 'chien', 'grenouille',
               'cheval', 'bateau', 'camion']

# Visualiser quelques images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(x_train_small[i])
    ax.set_title(class_names[y_train_small[i][0]])
    ax.axis('off')
plt.suptitle("Exemples CIFAR-10")
plt.tight_layout()
plt.show()

# ============================================================
# ÉTAPE 1 — Charger MobileNet pré-entraîné (sans la tête)
# ============================================================
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,     # ← On retire la dernière couche
    weights='imagenet'     # ← Poids pré-entraînés sur ImageNet
)

print(f"\nNombre de couches MobileNet : {len(base_model.layers)}")

# ============================================================
# ÉTAPE 2 — GELER les couches pré-entraînées
# ============================================================
base_model.trainable = False  # On ne touche pas aux poids existants

print(f"Paramètres entraînables    : "
      f"{sum([tf.size(w).numpy() for w in base_model.trainable_weights])}")
print(f"Paramètres gelés           : "
      f"{sum([tf.size(w).numpy() for w in base_model.non_trainable_weights])}")

# ============================================================
# ÉTAPE 3 — Ajouter notre nouvelle tête
# ============================================================
model_tl = models.Sequential([
    base_model,                          # Couches pré-entraînées (gelées)
    layers.GlobalAveragePooling2D(),     # Réduire les features
    layers.Dense(128, activation='relu'),# Notre couche cachée
    layers.Dropout(0.3),                 # Régularisation
    layers.Dense(10, activation='softmax')  # Notre sortie : 10 classes
])

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_tl.summary()

# ============================================================
# ÉTAPE 4 — Entraîner SEULEMENT la nouvelle tête
# ============================================================
print("\n Phase 1 : Entraînement de la tête uniquement")

history_tl = model_tl.fit(
    x_train_small, y_train_enc,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

loss_tl, acc_tl = model_tl.evaluate(
    x_test_small, y_test_enc, verbose=0
)
print(f"\nAccuracy Transfer Learning (phase 1) : {acc_tl*100:.2f}%")

# ============================================================
# ÉTAPE 5 — Fine-tuning : dégeler les dernières couches
# ============================================================
print("\n Phase 2 : Fine-tuning (dégel des dernières couches)")

# Dégeler les 30 dernières couches de MobileNet
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False  # On garde les premières couches gelées

# Recompiler avec un learning rate PLUS FAIBLE
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model_tl.fit(
    x_train_small, y_train_enc,
    epochs=5,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

loss_ft, acc_ft = model_tl.evaluate(
    x_test_small, y_test_enc, verbose=0
)
print(f"Accuracy après fine-tuning : {acc_ft*100:.2f}%")

# ============================================================
# ÉTAPE 6 — Comparaison avec modèle from scratch
# ============================================================
print("\n Modèle from scratch (sans Transfer Learning)")

model_scratch = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu',
                  input_shape=(32,32,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_scratch.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_scratch.fit(
    x_train_small, y_train_enc,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

_, acc_scratch = model_scratch.evaluate(
    x_test_small, y_test_enc, verbose=0
)

# ============================================================
# TABLEAU COMPARATIF
# ============================================================
print("\n" + "="*45)
print("COMPARAISON FINALE")
print("="*45)
print(f"From Scratch          : {acc_scratch*100:.2f}%")
print(f"Transfer Learning     : {acc_tl*100:.2f}%")
print(f"Après Fine-tuning     : {acc_ft*100:.2f}%")

print(f"""
POURQUOI TRANSFER LEARNING ÉCONOMISE DU TEMPS ?

MobileNet a déjà appris à reconnaître des formes
sur 1.2 million d'images — on réutilise ce savoir.
On entraîne seulement notre petite tête de sortie
au lieu de tout le réseau → beaucoup plus rapide.
Avec seulement 5000 images, Transfer Learning
dépasse souvent un modèle entraîné from scratch.
Le fine-tuning permet d'adapter aussi les dernières
couches à notre dataset